<a href="https://colab.research.google.com/github/vanamnagasaiVarshini/thiranex_Projects/blob/main/Secure_Login_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn sqlalchemy passlib bcrypt pyjwt python-multipart slowapi pydantic[email] nest-asyncio
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 4s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸npm notice
npm notice New major version of npm available! 10.8.2 -> 11.13.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.13.0
npm notice To update run: npm install -g npm@11.13.0
npm notice
⠸

In [3]:
%%writefile database.py
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, declarative_base
engine = create_engine("sqlite:///./secure_login.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()
def get_db():
    db = SessionLocal()
    try: yield db
    finally: db.close()

%%writefile models.py
from sqlalchemy import Column, Integer, String
from database import Base
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True, nullable=False)
    email = Column(String, unique=True, index=True, nullable=False)
    hashed_password = Column(String, nullable=False)

%%writefile schemas.py
import re
from pydantic import BaseModel, EmailStr, Field, field_validator
class UserCreate(BaseModel):
    username: str = Field(..., min_length=3, max_length=50)
    email: EmailStr
    password: str
    @field_validator('password')
    @classmethod
    def validate_password_strength(cls, value):
        if len(value) < 8 or not re.search(r"[A-Z]", value) or not re.search(r"\d", value) or not re.search(r"[!@#$%^&*()_+\-=\[\]{}|;:',.<>/?]", value):
            raise ValueError("Password requires 8+ chars, uppercase, digit, and symbol")
        return value
class UserLogin(BaseModel):
    username: str
    password: str
class UserResponse(BaseModel):
    id: int
    username: str
    email: str
    class Config: from_attributes = True

%%writefile limiter.py
from slowapi import Limiter
from slowapi.util import get_remote_address
limiter = Limiter(key_func=get_remote_address)

%%writefile auth.py
import os
from datetime import datetime, timedelta
from jose import JWTError, jwt
from passlib.context import CryptContext
from fastapi import HTTPException, status
from sqlalchemy.orm import Session
import models
SECRET_KEY = "super_secret_temporary_key"
ALGORITHM = "HS256"
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
def verify_password(plain, hashed): return pwd_context.verify(plain, hashed)
def get_password_hash(password): return pwd_context.hash(password)
def create_access_token(data: dict):
    to_encode = data.copy()
    to_encode.update({"exp": datetime.utcnow() + timedelta(minutes=30)})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
def get_user_from_token(token: str, db: Session):
    try:
        username: str = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM]).get("sub")
        if not username: raise JWTError
    except JWTError: raise HTTPException(status_code=401, detail="Invalid token")
    user = db.query(models.User).filter(models.User.username == username).first()
    if not user: raise HTTPException(status_code=401)
    return user

%%writefile routes.py
from fastapi import APIRouter, Depends, HTTPException, status, Response, Request
from sqlalchemy.orm import Session
import models, schemas, auth
from database import get_db
from limiter import limiter
router = APIRouter()
@router.post("/register", response_model=schemas.UserResponse, status_code=201)
def register(user: schemas.UserCreate, db: Session = Depends(get_db)):
    if db.query(models.User).filter((models.User.username==user.username)|(models.User.email==user.email)).first():
        raise HTTPException(status_code=400, detail="Username/email taken")
    new_user = models.User(username=user.username, email=user.email, hashed_password=auth.get_password_hash(user.password))
    db.add(new_user)
    db.commit()
    db.refresh(new_user)
    return new_user
@router.post("/login")
@limiter.limit("5/minute")
def login(request: Request, response: Response, user_credentials: schemas.UserLogin, db: Session = Depends(get_db)):
    user = db.query(models.User).filter(models.User.username == user_credentials.username).first()
    if not user or not auth.verify_password(user_credentials.password, user.hashed_password):
        raise HTTPException(status_code=401, detail="Incorrect credentials")
    access_token = auth.create_access_token(data={"sub": user.username})
    response.set_cookie(key="access_token", value=f"Bearer {access_token}", httponly=True, max_age=1800, samesite="lax")
    return {"message": "Success"}
def get_current_user(request: Request, db: Session = Depends(get_db)):
    token = request.cookies.get("access_token")
    if not token: raise HTTPException(status_code=401, detail="Not authenticated")
    return auth.get_user_from_token(token.partition(" ")[2], db)
@router.get("/profile", response_model=schemas.UserResponse)
def get_profile(current_user: models.User = Depends(get_current_user)): return current_user
@router.get("/dashboard")
def get_dashboard(current_user: models.User = Depends(get_current_user)): return {"message": f"Welcome, {current_user.username}!"}
@router.post("/logout")
def logout(response: Response):
    response.delete_cookie(key="access_token", httponly=True, samesite="lax")
    return {"message": "Logged out"}

%%writefile main.py
from fastapi import FastAPI, Depends, Request
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from slowapi.errors import RateLimitExceeded
from slowapi import _rate_limit_exceeded_handler
import models
from database import engine
from routes import router
from limiter import limiter
models.Base.metadata.create_all(bind=engine)
app = FastAPI()
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
app.include_router(router, prefix="/api")
@app.get("/", response_class=HTMLResponse)
def read_root():
    with open("templates/index.html") as f: return f.read()

Overwriting database.py


In [5]:
!mkdir -p templates

In [6]:
%%writefile templates/index.html
<!DOCTYPE html>
<html><head><title>Secure Login System</title>
<style>
body { font-family: 'Inter', sans-serif; background-color: #1a1a2e; color: #fff; display: flex; justify-content: center; align-items: center; height: 100vh; margin: 0; }
.container { background-color: #16213e; padding: 30px; border-radius: 10px; width: 350px; text-align: center; }
input { width: 100%; padding: 10px; margin: 10px 0; border-radius: 5px; box-sizing: border-box; }
button { background-color: #0f3460; color: #fff; padding: 10px; border: none; border-radius: 5px; cursor: pointer; width: 100%; margin-top: 10px; }
.hidden { display: none; }
</style>
</head><body>
<div class="container">
    <div id="auth-view">
        <h2 id="form-title">Secure Login</h2>
        <form id="auth-form">
            <input type="text" id="username" placeholder="Username" required>
            <input type="email" id="email" placeholder="Email" class="hidden">
            <input type="password" id="password" placeholder="Password" required>
            <button type="submit" id="submit-btn" style="margin-top: 15px;">Login</button>
        </form>
        <p style="font-size: 0.8em; cursor: pointer; color: #8892b0;" onclick="toggleView()" id="toggle-link">Switch to Register</p>
        <div id="auth-message" style="color: #e94560;"></div>
    </div>
    <div id="dashboard-view" class="hidden">
        <h2 style="color: #e94560;">Dashboard</h2><p id="dashboard-welcome"></p>
        <button onclick="fetchProfile()">Load Profile Data</button>
        <pre id="profile-data" style="background:#0f3460; padding:10px; border-radius:5px;text-align:left;"></pre>
        <button onclick="logout()" style="background-color: #e94560;">Logout Securely</button>
    </div>
</div>
<script>
    let isLogin = true;
    function toggleView() {
        isLogin = !isLogin;
        document.getElementById('form-title').innerText = isLogin ? 'Secure Login' : 'Register';
        document.getElementById('submit-btn').innerText = isLogin ? 'Login' : 'Register';
        document.getElementById('email').classList.toggle('hidden', isLogin);
    }
    document.getElementById('auth-form').addEventListener('submit', async (e) => {
        e.preventDefault();
        const u = document.getElementById('username').value, p = document.getElementById('password').value, em = document.getElementById('email').value;
        const res = await fetch(isLogin ? '/api/login' : '/api/register', { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(isLogin ? {username:u, password:p} : {username:u, email:em, password:p}) });
        if(res.ok) { isLogin ? showDashboard() : location.reload(); }
        else { document.getElementById('auth-message').innerText = "Validation failed. Try a stronger password."; }
    });
    async function showDashboard() {
        document.getElementById('auth-view').classList.add('hidden'); document.getElementById('dashboard-view').classList.remove('hidden');
        const res = await fetch('/api/dashboard');
        if(res.ok) { document.getElementById('dashboard-welcome').innerText = (await res.json()).message; }
    }
    async function fetchProfile() {
        const res = await fetch('/api/profile');
        if(res.ok) { const d = await res.json(); document.getElementById('profile-data').innerText = `User ID: ${d.id}\nEmail: ${d.email}`; }
    }
    async function logout() { await fetch('/api/logout', {method:'POST'}); location.reload(); }
</script>
</body></html>

Writing templates/index.html


In [8]:
import urllib

# Fetch and print the Endpoint IP (this acts as the Localtunnel password!)
print("--- YOUR LOCALTUNNEL PASSWORD IS BELOW ---")
print(urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode('utf8').strip())
print("------------------------------------------")

# Start FastAPI/Uvicorn in the background using Colab's bash utility
get_ipython().system_raw('uvicorn main:app --host 0.0.0.0 --port 8000 &')

# Start localtunnel in the background
get_ipython().system_raw('lt --port 8000 >> url.txt 2>&1 &')

# Wait a brief moment to make sure the URL file populates, then print it
import time
time.sleep(3)
!cat url.txt

print("\n=> Click the 'loca.lt' link above, and enter the password when prompted! <=\n")

--- YOUR LOCALTUNNEL PASSWORD IS BELOW ---
34.23.16.76
------------------------------------------
your url is: https://bumpy-moons-act.loca.lt
your url is: https://poor-mice-refuse.loca.lt

=> Click the 'loca.lt' link above, and enter the password when prompted! <=

